# 05 Figures: metrics and capture curve

Drafted with an AI assistant, see docs/ai_log.md.

Inputs: `data/processed/model_table.parquet`, `data/processed/predictions.parquet`.
Outputs: `figures/metrics.csv`, `figures/table_model_gaps.tex`, `figures/capture_curve.pdf` (the three main models only; the heating sensitivity models are in `metrics.csv` and the gap table).

Metrics per model, split and label: capture at 5% and 10%, ROC AUC, average precision. A `random` row gives the reference values. Test rows also carry 95% bootstrap intervals (`<metric>_lo`, `<metric>_hi`).

In [ ]:
import helpers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
model_table = pd.read_parquet(helpers.PROCESSED / "model_table.parquet")
predictions = pd.read_parquet(helpers.PROCESSED / "predictions.parquet")
data = model_table.merge(predictions, on=["egid", "cutoff", "split"], how="inner")
assert len(data) == len(predictions)
# Fixed row order before scoring: ties in a score (many buildings share the same
# neighbour share) are broken by this order in the stable sorts below.
data = data.sort_values(["split", "model", "egid"]).reset_index(drop=True)
helpers.FIGURES.mkdir(exist_ok=True)

Bootstrap on the test split. Resampling unit: whole municipalities (`ggdenr`), because neighbour shares and outcomes are correlated within a municipality (docs/decisions.md). B = 200, seed 0. The replicate row indices are drawn once and used for every model, so differences between models are paired. Rows of each model are in egid order (same buildings, same order), a municipality drawn twice contributes its rows twice, and the copies of a row sit next to each other in the replicate; the stable sort in the capture metric then breaks ties as in the original data.

In [ ]:
N_BOOT = 200
SEED = 0
METRICS = ["capture_5", "capture_10", "roc_auc", "avg_precision"]


def score_metrics(y, score):
    return {
        "capture_5": helpers.capture_at_k(y, score, 0.05),
        "capture_10": helpers.capture_at_k(y, score, 0.10),
        "roc_auc": roc_auc_score(y, score),
        "avg_precision": average_precision_score(y, score),
    }


test_part = data[data["split"] == "test"]
models = sorted(test_part["model"].unique())
# Same buildings in the same order for every model, so one set of indices fits all
egids = test_part[test_part["model"] == models[0]]["egid"].to_numpy()
for model in models:
    assert np.array_equal(test_part[test_part["model"] == model]["egid"].to_numpy(), egids)
muni = test_part[test_part["model"] == models[0]]["ggdenr"].to_numpy()
replicates = helpers.cluster_bootstrap_indices(muni, N_BOOT, SEED)
sizes = [len(r) for r in replicates]
print(f"{len(set(muni))} municipalities in the test pool")
print(f"replicate size {min(sizes)} to {max(sizes)} rows")

# boot[(model, label)][metric] = array with one value per replicate
boot = {}
for model in models:
    part = test_part[test_part["model"] == model]
    score = part["score"].to_numpy()
    for label in ["y", "y_2024"]:
        y = part[label].astype(int).to_numpy()
        values = [score_metrics(y[idx], score[idx]) for idx in replicates]
        boot[(model, label)] = {m: np.array([v[m] for v in values]) for m in METRICS}

One row per (model, split, label). Labels: `y` for both splits, `y_2024` for the test split. Test rows get the 2.5th and 97.5th percentiles of the replicate values in `<metric>_lo` and `<metric>_hi`; train and `random` rows leave them empty.

In [ ]:
rows = []
for split in ["train", "test"]:
    labels = ["y"] if split == "train" else ["y", "y_2024"]
    for label in labels:
        for model, part in data[data["split"] == split].groupby("model"):
            y = part[label].astype(int).to_numpy()
            score = part["score"].to_numpy()
            rows.append(
                {
                    "model": model,
                    "split": split,
                    "label": label,
                    "n": len(y),
                    "positives": int(y.sum()),
                    "base_rate": y.mean(),
                    "capture_5": helpers.capture_at_k(y, score, 0.05),
                    "capture_10": helpers.capture_at_k(y, score, 0.10),
                    "roc_auc": roc_auc_score(y, score),
                    "avg_precision": average_precision_score(y, score),
                }
            )
        # Random reference, once per (split, label): same pool, expected values.
        # The last model row has the same n, positives and base rate as all others.
        last = rows[-1]
        rows.append(
            {
                "model": "random",
                "split": split,
                "label": label,
                "n": last["n"],
                "positives": last["positives"],
                "base_rate": last["base_rate"],
                "capture_5": 0.05,
                "capture_10": 0.10,
                "roc_auc": 0.5,
                "avg_precision": last["base_rate"],
            }
        )
metrics = pd.DataFrame(rows).sort_values(["split", "label", "model"]).reset_index(drop=True)
assert not metrics.duplicated(["model", "split", "label"]).any()
# Bootstrap intervals on the test rows (percentiles of the replicate values)
for metric in METRICS:
    metrics[f"{metric}_lo"] = np.nan
    metrics[f"{metric}_hi"] = np.nan
for (model, label), reps in boot.items():
    row = (metrics["model"] == model) & (metrics["split"] == "test") & (metrics["label"] == label)
    for metric in METRICS:
        metrics.loc[row, f"{metric}_lo"] = np.percentile(reps[metric], 2.5)
        metrics.loc[row, f"{metric}_hi"] = np.percentile(reps[metric], 97.5)
metrics = metrics.round(6)
metrics.to_csv(helpers.FIGURES / "metrics.csv", index=False, lineterminator="\n")
print(metrics.to_string())

Paired gaps on the test split: the difference between two models on the same buildings, replicate by replicate with the same municipality draws, for the headline pairs among the models present. Point estimate from the full test pool, 95% percentile interval of the replicate differences. Output: `figures/table_model_gaps.tex`.

In [ ]:
GAP_METRICS = {"capture_10": "Capture at 10%", "roc_auc": "ROC AUC"}
PAIRS = [
    ("lasso", "nbr_share"),
    ("lasso", "roof_yield"),
    ("nbr_share", "roof_yield"),
    # Heating sensitivity models against the main LASSO (leakage-contaminated upper bound)
    ("lasso_heat_source", "lasso"),
    ("lasso_heat_date", "lasso"),
]
PAIRS = [(a, b) for a, b in PAIRS if a in models and b in models]
LABELS = {"y": "Test, first PV 2024-2025", "y_2024": "Test, first PV 2024 only"}


def full_value(model, label, metric):
    part = test_part[test_part["model"] == model]
    return score_metrics(part[label].astype(int).to_numpy(), part["score"].to_numpy())[metric]


# Plain LaTeX tabular with fixed formatting, to be \\input in the paper
lines = [
    "\\begin{tabular}{llrr}",
    "\\toprule",
    "Label & Gap (first minus second) & Capture at 10\\% & ROC AUC \\\\",
    "\\midrule",
]
for label, label_text in LABELS.items():
    for a, b in PAIRS:
        cells = []
        for metric in GAP_METRICS:
            point = full_value(a, label, metric) - full_value(b, label, metric)
            diff = boot[(a, label)][metric] - boot[(b, label)][metric]
            lo, hi = np.percentile(diff, [2.5, 97.5])
            cells.append(f"{point:.3f} [{lo:.3f}, {hi:.3f}]")
        name = f"{a.replace('_', ' ')} minus {b.replace('_', ' ')}"
        lines.append(f"{label_text} & {name} & " + " & ".join(cells) + " \\\\")
    if label != list(LABELS)[-1]:
        lines.append("\\midrule")
lines += ["\\bottomrule", "\\end{tabular}"]
(helpers.FIGURES / "table_model_gaps.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

Capture curve on the test split, label `y`: share of adopters found as we go down the ranking.

In [ ]:
# Main models only: the heating models use leaked information and must not look like candidates
CURVE_MODELS = ["roof_yield", "nbr_share", "lasso"]
test = data[(data["split"] == "test") & data["model"].isin(CURVE_MODELS)]
fig, ax = plt.subplots(figsize=(5, 4))
for model, part in test.groupby("model"):
    y = part["y"].astype(int).to_numpy()
    order = np.argsort(-part["score"].to_numpy(), kind="stable")
    share_ranked = np.arange(1, len(y) + 1) / len(y)
    share_captured = np.cumsum(y[order]) / y.sum()
    ax.plot(share_ranked, share_captured, label=model)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="random")
ax.set_xlabel("Share of buildings ranked (highest score first)")
ax.set_ylabel("Share of adopters captured")
ax.set_title("Capture curve, test split (first PV 2024-2025)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(helpers.FIGURES / "capture_curve.pdf", metadata={"CreationDate": None})
plt.close(fig)